# PART 8 スターターNotebook
## Agentic AIで動画制作を回す

テーマを与えると1分の解説動画を完成させる、制作ワークフローの雛形です。

---

### あなたが設計して書き換えるのは、次の4つだけです

| | 場所 | 決めること |
|---|---|---|
| ① | `VideoState` | 状態に何を持たせるか |
| ② | `check_video` | 機械が判定するのは何か |
| ③ | `review_plan` / `review_final` | どこで人間に止めるか |
| ④ | `route_after_check` | 落ちたとき、どの工程へ戻すか |

ffprobeの呼び出し、字幕の生成、ffmpegの組み立て、グラフの雛形は用意してあります。
**動画処理のデバッグに時間を使わないでください。**この課題で見るのは、仕事の分解と承認の位置です。

---

### 2つのルート

- **標準課題**：LLMを使わない。テンプレートとルールで工程を通す「制作ワークフロー」
- **発展課題**：企画・台本・絵コンテのノードにLLMを接続した「制作Agent」


> **最初の1回は必ず検査に落ちます。**初期設定では動画が約70秒になり、55〜65秒の条件を満たしません。
> これは仕込みです。落ちてから、どこへ戻すかを設計するのがこの課題です。
標準課題だけでも課題の目的は達成できます。最後のセルに発展の入口があります。


## 0. セットアップ

最初に1回だけ実行します。2〜3分かかります。


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg fonts-noto-cjk > /dev/null
!pip -q install langgraph
print('setup done')


In [ ]:
import json, subprocess, textwrap
from pathlib import Path
from typing import TypedDict

WORK = Path('/content/work')
(WORK / 'shots').mkdir(parents=True, exist_ok=True)

# 日本語フォントを探す（見つからなければ英語で作る）
FONT_CANDIDATES = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
    '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
]
FONT_PATH = next((p for p in FONT_CANDIDATES if Path(p).exists()), None)
print('font:', FONT_PATH)


## 1. 状態（① あなたが書き換えるところ）

ワークフローの各ノードは、この `VideoState` を受け取って、更新したものを返します。
PART 7の `State` と構造は同じです。工程が増えただけです。

**TODO ①**：自分の設計に必要な項目を足してください。何を足したか、提出時に説明できるようにしておくこと。


In [ ]:
class VideoState(TypedDict, total=False):
    # --- 入力 ---
    theme: str               # テーマ
    sources: list[str]       # 学生が入力する出典3本
    caption_only: bool       # True なら音声なし・字幕のみ

    # --- 制作の途中 ---
    points: list[str]        # 要点3つ
    script: str              # 台本
    shots: list[dict]        # 絵コンテ [{'text':..., 'seconds':...}]
    subtitle_path: str
    audio_path: str
    video_path: str

    # --- 判定と履歴 ---
    check_report: dict
    attempts: int
    approved: bool
    feedback: str            # 人間からの指摘

    # TODO ①: 必要な項目をここに足す

MAX_ATTEMPTS = 3


## 2. ツール（提供。読むだけでよい）

字幕、スライド、音声、組み立て、検査。ここは触らなくても課題は完成します。


In [ ]:
from PIL import Image, ImageDraw, ImageFont

W, H = 1280, 720


def make_slide(index: int, text: str) -> str:
    """1カット分の静止画を作る。"""
    img = Image.new('RGB', (W, H), (32, 34, 31))
    draw = ImageDraw.Draw(img)
    font = ImageFont.truetype(FONT_PATH, 54) if FONT_PATH else ImageFont.load_default()
    lines = textwrap.wrap(text, width=18) or ['']
    y = H // 2 - len(lines) * 36
    for line in lines:
        w = draw.textlength(line, font=font)
        draw.text(((W - w) / 2, y), line, font=font, fill=(247, 245, 238))
        y += 72
    path = WORK / 'shots' / f'shot_{index:02d}.png'
    img.save(path)
    return str(path)


def make_subtitles(shots: list[dict]) -> str:
    """絵コンテからSRT字幕を作る。"""
    def ts(sec: float) -> str:
        h, rem = divmod(int(sec), 3600)
        m, s = divmod(rem, 60)
        ms = int((sec - int(sec)) * 1000)
        return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'

    out, t = [], 0.0
    for i, shot in enumerate(shots, start=1):
        end = t + shot['seconds']
        out.append(f"{i}\n{ts(t)} --> {ts(end)}\n{shot['text']}\n")
        t = end
    path = WORK / 'subtitles.srt'
    path.write_text('\n'.join(out), encoding='utf-8')
    return str(path)


def make_silent_audio(seconds: float) -> str:
    """無音トラックを作る（caption_only=False のとき使う）。"""
    path = WORK / 'narration.m4a'
    subprocess.run(['ffmpeg', '-y', '-f', 'lavfi', '-i', 'anullsrc=r=44100:cl=stereo',
                    '-t', str(seconds), '-c:a', 'aac', str(path)],
                   capture_output=True, check=True)
    return str(path)


In [ ]:
def assemble_video(shots: list[dict], subtitle_path: str,
                   audio_path: str | None = None) -> str:
    """画像・字幕・音声を1本のmp4にする。"""
    clips = []
    for i, shot in enumerate(shots):
        img = make_slide(i, shot['text'])
        clip = WORK / 'shots' / f'clip_{i:02d}.mp4'
        subprocess.run(['ffmpeg', '-y', '-loop', '1', '-i', img,
                        '-t', str(shot['seconds']), '-r', '30',
                        '-vf', f'scale={W}:{H}', '-pix_fmt', 'yuv420p', str(clip)],
                       capture_output=True, check=True)
        clips.append(clip)

    listfile = WORK / 'shots.txt'
    listfile.write_text(''.join(f"file '{c}'\n" for c in clips), encoding='utf-8')

    joined = WORK / 'joined.mp4'
    subprocess.run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0',
                    '-i', str(listfile), '-c', 'copy', str(joined)],
                   capture_output=True, check=True)

    out = WORK / 'output.mp4'
    cmd = ['ffmpeg', '-y', '-i', str(joined)]
    if audio_path:
        cmd += ['-i', audio_path, '-c:a', 'aac', '-shortest']
    # 字幕の焼き込み。libassが無い環境では失敗するので、その場合は字幕なしで出す
    try:
        subprocess.run(cmd + ['-vf', f'subtitles={subtitle_path}',
                              '-c:v', 'libx264', '-pix_fmt', 'yuv420p', str(out)],
                       capture_output=True, check=True)
    except subprocess.CalledProcessError:
        print('字幕の焼き込みに失敗しました。スライドの文字だけで出力します。')
        subprocess.run(cmd + ['-c:v', 'libx264', '-pix_fmt', 'yuv420p', str(out)],
                       capture_output=True, check=True)
    return str(out)


def inspect_video(path: str) -> dict:
    """ffprobeで尺・解像度・音声の有無を取得する。"""
    r = subprocess.run(['ffprobe', '-v', 'quiet', '-print_format', 'json',
                        '-show_format', '-show_streams', path],
                       capture_output=True, text=True, check=True)
    info = json.loads(r.stdout)
    streams = info.get('streams', [])
    return {
        'duration': float(info['format']['duration']),
        'has_audio': any(s['codec_type'] == 'audio' for s in streams),
        'width': next((s.get('width') for s in streams
                       if s['codec_type'] == 'video'), None),
    }


## 3. 入力

**標準課題では、出典は自分で3本入力します。**ワークフローがWeb検索で事実を作ることはしません。
発展課題では検索やMCPで候補を集めても構いませんが、**最後の事実確認は人間が行います。**


In [ ]:
THEME = 'ムンバイの通勤鉄道が都市にもたらしたもの'

SOURCES = [
    '（出典1をここに書く：URLまたは書誌情報）',
    '（出典2）',
    '（出典3）',
]

CAPTION_ONLY = True   # True: 音声なし・字幕のみ / False: 無音トラックを付ける


## 4. 制作ノード（提供。標準課題ではテンプレートで動く）

`plan_points` と `write_script` が、発展課題でLLMに置き換わる場所です。
標準課題では、あなたが書いた要点をそのまま使います。


In [ ]:
# 注意: この初期値では、動画が約70秒になります。
# check_video の 55〜65秒に収まりません。1回目は必ず落ちます。
# それが正常です。落ちてから、どこへ戻すかを設計してください。
POINTS = [
    '要点1をここに書く',
    '要点2をここに書く',
    '要点3をここに書く',
]

SECONDS_PER_POINT = 23.5   # ← 3点で70.5秒。ここも設計対象です


def plan_points(state: VideoState) -> VideoState:
    # 発展課題では、ここをLLMの呼び出しに置き換える
    return {**state, 'points': POINTS, 'attempts': 0}


def check_sources(state: VideoState) -> VideoState:
    """出典のない要点は先へ進ませない。"""
    if len([s for s in state['sources'] if s and not s.startswith('（')]) < 3:
        raise ValueError('出典を3本入力してください。推測で先へ進めません。')
    return state


def write_script(state: VideoState) -> VideoState:
    # 発展課題では、ここをLLMの呼び出しに置き換える
    script = '。'.join(state['points']) + '。'
    return {**state, 'script': script}


def make_storyboard(state: VideoState) -> VideoState:
    shots = [{'text': p, 'seconds': SECONDS_PER_POINT} for p in state['points']]
    return {**state, 'shots': shots}


def build_video(state: VideoState) -> VideoState:
    sub = make_subtitles(state['shots'])
    total = sum(s['seconds'] for s in state['shots'])
    audio = None if state['caption_only'] else make_silent_audio(total)
    path = assemble_video(state['shots'], sub, audio)
    return {**state, 'subtitle_path': sub, 'audio_path': audio or '',
            'video_path': path, 'attempts': state.get('attempts', 0) + 1}


## 5. 検査条件（② あなたが書き換えるところ）

**機械が判定できることだけ**をここに書きます。
「面白いか」「分かりやすいか」は入りません。それは人間の判断で、6章の `review` に置きます。

**TODO ②**：判定項目を足してください。最低でも1つは自分で考えたものを入れること。


In [ ]:
def check_video(state: VideoState) -> VideoState:
    info = inspect_video(state['video_path'])
    report = {
        'duration': round(info['duration'], 1),
        'duration_ok': 55 <= info['duration'] <= 65,
        'has_audio': info['has_audio'],
        # TODO ②: 判定項目を足す
        # 例) 字幕の総文字数が台本と一致するか、カット数が3以上か
    }
    report['passed'] = (
        report['duration_ok']
        and (info['has_audio'] or state['caption_only'])
    )
    print('check:', report)
    return {**state, 'check_report': report}


## 6. 承認の位置（③ あなたが書き換えるところ）

雛形では2か所で止めます。企画の後（作り始める前）と、完成後（公開する前）です。

**TODO ③**：この2か所でよいか考えてください。増やしても減らしてもかまいませんが、
**なぜそこで止めるのかを説明できること。**判断の基準はPART 7.4.2です。
取り消せない操作の前、影響範囲が広がるとき、行き詰まったとき、外部に出るとき。


In [ ]:
from langgraph.types import interrupt, Command


def review_plan(state: VideoState) -> VideoState:
    """企画と事実確認の承認。ここから先は作り直しが高くつく。"""
    answer = interrupt({
        'points': state['points'],
        'sources': state['sources'],
        'script': state['script'],
        'question': '企画と事実確認は妥当ですか？',
        'choices': ['approve', 'revise'],
    })
    return {**state,
            'approved': answer['decision'] == 'approve',
            'feedback': answer.get('feedback', '')}


def review_final(state: VideoState) -> VideoState:
    """公開判断。8.6の観点をここで確認する。"""
    answer = interrupt({
        'video_path': state['video_path'],
        'check_report': state['check_report'],
        'question': '公開してよいですか？（誤情報・著作権・肖像・生成物表示）',
        'choices': ['approve', 'revise', 'stop'],
    })
    return {**state, 'approved': answer['decision'] == 'approve',
            'feedback': answer.get('feedback', '')}


## 7. 戻り先（④ あなたが書き換えるところ）

落ちたとき、**全部やり直すのではありません。**原因に応じて戻る場所を変えます。

**TODO ④**：`route_after_check` の分岐を設計してください。
尺が長いなら台本へ、音がないなら組み立てへ、といった対応です。


In [ ]:
from langgraph.graph import StateGraph, START, END


def route_after_plan(state: VideoState) -> str:
    return 'storyboard' if state['approved'] else 'plan'


def route_after_check(state: VideoState) -> str:
    r = state['check_report']
    if r['passed']:
        return 'review_final'
    if state['attempts'] >= MAX_ATTEMPTS:
        return 'give_up'
    # TODO ④: 原因ごとに戻り先を変える
    if not r['duration_ok']:
        return 'script'      # 尺が合わない → 台本から
    return 'build'           # それ以外 → 組み立て直し


def route_after_final(state: VideoState) -> str:
    return END if state['approved'] else 'build'


def give_up(state: VideoState) -> VideoState:
    print(f"{MAX_ATTEMPTS}回試しても通りませんでした。人間に引き継ぎます。")
    print(state['check_report'])
    return state


## 8. 組み立てて実行する


In [ ]:
from langgraph.checkpoint.memory import MemorySaver

g = StateGraph(VideoState)
g.add_node('plan', plan_points)
g.add_node('sources', check_sources)
g.add_node('script', write_script)
g.add_node('review_plan', review_plan)
g.add_node('storyboard', make_storyboard)
g.add_node('build', build_video)
g.add_node('check', check_video)
g.add_node('review_final', review_final)
g.add_node('give_up', give_up)

g.add_edge(START, 'plan')
g.add_edge('plan', 'sources')
g.add_edge('sources', 'script')
g.add_edge('script', 'review_plan')
g.add_conditional_edges('review_plan', route_after_plan)
g.add_edge('storyboard', 'build')
g.add_edge('build', 'check')
g.add_conditional_edges('check', route_after_check)
g.add_conditional_edges('review_final', route_after_final)
g.add_edge('give_up', END)

app = g.compile(checkpointer=MemorySaver())
config = {'configurable': {'thread_id': 'video-001'}}

result = app.invoke({'theme': THEME, 'sources': SOURCES,
                     'caption_only': CAPTION_ONLY}, config)
print('--- 停止しました。上の内容を確認してください ---')
print(result.get('__interrupt__', result))


## 9. 承認して再開する

内容を読んでから実行します。`revise` を選ぶときは、**何を直してほしいかを書いてください。**
その指摘が次の工程に渡ります。


In [ ]:
# 企画を承認して先へ進む
result = app.invoke(Command(resume={'decision': 'approve'}), config)
print(result.get('__interrupt__', result))


In [ ]:
# 直してほしい場合はこちら
# result = app.invoke(Command(resume={
#     'decision': 'revise',
#     'feedback': '要点2が出典の内容と合っていません',
# }), config)


In [ ]:
from IPython.display import Video
Video('/content/work/output.mp4', embed=True, width=720)


## 10. 提出物チェック

- [ ] 完成した `output.mp4`
- [ ] このNotebook（グラフ、ノード、承認の位置が読める状態で）
- [ ] 工程のログ。どこで落ち、どこへ戻り、何回で通ったか
- [ ] 人間が承認・却下した箇所と、その理由
- [ ] 8.6の表（誤情報・著作権・肖像・生成物表示・コスト・ログ）の確認結果

**修正ループを最低1回は回してください。**1発で通った場合は、`check_video` のしきい値を厳しくして、
意図的に落として戻る様子を記録してください。この課題で見たいのは、失敗したときに何が起きるかです。


## 11. 発展：LLMを接続する

`plan_points` と `write_script` を、LLMの呼び出しに置き換えます。
それ以外のノード、承認の位置、検査条件は**そのまま使えます。**設計は変わりません。

```python
# 例：OpenAI互換のエンドポイントを使う場合
# from openai import OpenAI
# client = OpenAI(api_key=userdata.get('API_KEY'))
#
# def plan_points(state):
#     prompt = (f"テーマ: {state['theme']}\n"
#               f"出典:\n" + '\n'.join(state['sources']) + '\n\n'
#               'この出典だけを根拠に、要点を3つ、各40字以内で挙げてください。'
#               '出典に書かれていないことは書かないでください。')
#     ...
```

**注意**：LLMを繋いでも、`check_sources` と `review_plan` は外さないでください。
PART 1で見たとおり、出典を渡しても、それらしい数字と固有名詞は混ざります。
**最後の事実確認は人間が行います。**
